In [4]:
"""
fetch_dasu_monitorings.py  (паралельна версія, x10 швидше)
===========================================================
Завантажує усі моніторинги ДАСУ з Prozorro Audit API.

Стратегія:
    1. Швидко (1 потік) проходимо список моніторингів — це обов'язково
       послідовно, бо API віддає сторінки через offset.
    2. Збираємо ID всіх моніторингів у пам'ять.
    3. Паралельно (N потоків) тягнемо деталі кожного моніторингу.
       Тут і відбувається прискорення — деталі незалежні одна від одної.

Запуск:
    python fetch_dasu_monitorings.py

При перерві — перезапуск продовжить з місця де зупинився
(стан зберігається у dasu_progress.json).

Файли:
    dasu_monitorings.parquet  — повні дані моніторингів
    dasu_labels.parquet       — мітки для JOIN з тендерами
    dasu_ids.json             — список усіх ID (кеш етапу 1)
"""

import json
import os
import time
import urllib.request
import urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════
# НАЛАШТУВАННЯ
# ══════════════════════════════════════════════════════════════

API_BASE          = "https://audit-api.prozorro.gov.ua/api/2.5"
LIST_ENDPOINT     = f"{API_BASE}/monitorings"
DETAIL_ENDPOINT   = f"{API_BASE}/monitorings/{{}}"

PAGE_LIMIT        = 1000        # максимум у API — 1000 на сторінку
N_WORKERS         = 20          # паралельних запитів. 10-30 безпечно.
TIMEOUT           = 30
MAX_RETRIES       = 3
SAVE_EVERY        = 2000        # зберігати кожні N успішних деталей

OUTPUT_DETAILS    = "dasu_monitorings.parquet"
OUTPUT_LABELS     = "dasu_labels.parquet"
IDS_CACHE         = "dasu_ids.json"
PROGRESS_FILE     = "dasu_progress.json"


# ══════════════════════════════════════════════════════════════
# HTTP-обгортка
# ══════════════════════════════════════════════════════════════

def http_get_json(url: str, retries: int = MAX_RETRIES) -> dict:
    """GET → JSON з повторами при помилках."""
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 (research-script)"},
            )
            with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except (urllib.error.URLError, urllib.error.HTTPError,
                TimeoutError, json.JSONDecodeError) as e:
            last_err = e
            time.sleep(min(2 ** attempt, 10))
    raise RuntimeError(f"Не вдалося отримати {url}: {last_err}")


# ══════════════════════════════════════════════════════════════
# ЕТАП 1 — Збираємо всі ID моніторингів (послідовно через пагінацію)
# ══════════════════════════════════════════════════════════════

def collect_all_ids() -> list[str]:
    """Проходить пагінацію і збирає всі ID моніторингів."""
    if os.path.exists(IDS_CACHE):
        with open(IDS_CACHE, encoding="utf-8") as f:
            ids = json.load(f)
        print(f"[Етап 1] З кешу: {len(ids):,} ID моніторингів")
        return ids

    print(f"[Етап 1] Збираємо ID через пагінацію (limit={PAGE_LIMIT}/сторінка)")
    ids = []
    # opt_fields=status — щоб отримати мінімум полів і не вантажити трафік
    next_url = f"{LIST_ENDPOINT}?limit={PAGE_LIMIT}&opt_fields=status"

    with tqdm(desc="Сторінки списку", unit=" page") as pbar:
        while next_url:
            page = http_get_json(next_url)
            data = page.get("data") or []
            if not data:
                break

            for item in data:
                mid = item.get("id")
                if mid:
                    ids.append(mid)

            pbar.update(1)
            pbar.set_postfix(ids=len(ids))

            next_page = page.get("next_page") or {}
            new_url   = next_page.get("uri")
            if new_url and new_url != next_url:
                next_url = new_url
            else:
                break

    with open(IDS_CACHE, "w", encoding="utf-8") as f:
        json.dump(ids, f)
    print(f"[Етап 1] Збережено {len(ids):,} ID у {IDS_CACHE}")
    return ids


# ══════════════════════════════════════════════════════════════
# ЕТАП 2 — Паралельне завантаження деталей
# ══════════════════════════════════════════════════════════════

def parse_monitoring(m: dict) -> dict:
    """Витягує плоский набір полів з повного моніторингу."""
    m_id          = m.get("id")
    tender_id     = m.get("tender_id")
    status        = m.get("status")
    date_created  = m.get("dateCreated")
    date_modified = m.get("dateModified")

    reasons = m.get("reasons") or []
    reasons_str = ",".join([str(r) for r in reasons]) if isinstance(reasons, list) else str(reasons)

    party = m.get("parties") or []
    procuring_entity_id = None
    procuring_entity_name = None
    for p in party:
        if "procuringEntity" in (p.get("roles") or []) or p.get("identifier"):
            ident = p.get("identifier") or {}
            procuring_entity_id   = ident.get("id")
            procuring_entity_name = p.get("name")
            break

    conclusion = m.get("conclusion") or {}
    has_conclusion     = bool(conclusion)
    violation_occurred = conclusion.get("violationOccurred")
    vtypes = conclusion.get("violationType") or []
    violation_types_str = ",".join([str(v) for v in vtypes]) if isinstance(vtypes, list) else str(vtypes)
    description     = conclusion.get("description")
    conclusion_date = conclusion.get("date")

    decision      = m.get("decision") or {}
    decision_date = decision.get("date")

    elimination = m.get("eliminationReport") or {}
    elimination_was_done   = bool(elimination)
    elimination_resolution = elimination.get("resolution")

    cancellation = m.get("cancellation") or {}
    cancellation_date = cancellation.get("date")

    appeal           = m.get("appeal") or {}
    appeal_was_filed = bool(appeal)
    appeal_date      = appeal.get("dateProceedings")

    liabilities   = m.get("liabilities") or []
    n_liabilities = len(liabilities) if isinstance(liabilities, list) else 0

    return {
        "monitoring_id":          m_id,
        "tender_id":              tender_id,
        "status":                 status,
        "reasons":                reasons_str,
        "date_created":           date_created,
        "date_modified":          date_modified,
        "procuring_entity_id":    procuring_entity_id,
        "procuring_entity_name":  procuring_entity_name,
        "has_conclusion":         has_conclusion,
        "violation_occurred":     violation_occurred,
        "violation_types":        violation_types_str,
        "conclusion_description": description[:500] if description else None,
        "conclusion_date":        conclusion_date,
        "decision_date":          decision_date,
        "elimination_was_done":   elimination_was_done,
        "elimination_resolution": elimination_resolution,
        "cancellation_date":      cancellation_date,
        "appeal_was_filed":       appeal_was_filed,
        "appeal_date":            appeal_date,
        "n_liabilities":          n_liabilities,
    }


def fetch_one(mid: str) -> dict | None:
    """Одна задача для пула потоків."""
    try:
        detail = http_get_json(DETAIL_ENDPOINT.format(mid))
        m_data = detail.get("data") or detail
        return parse_monitoring(m_data)
    except RuntimeError:
        return None


def fetch_details_parallel(ids: list[str]) -> pd.DataFrame:
    """Паралельне завантаження деталей через ThreadPoolExecutor."""
    # Завантажуємо вже зібрані щоб не качати повторно
    if os.path.exists(OUTPUT_DETAILS):
        existing = pd.read_parquet(OUTPUT_DETAILS)
        done_ids = set(existing["monitoring_id"].astype(str).tolist())
        rows     = existing.to_dict("records")
        print(f"[Етап 2] З кешу: {len(rows):,} моніторингів")
    else:
        done_ids = set()
        rows     = []

    # Залишаємо тільки ті ID яких ще немає
    todo = [mid for mid in ids if mid not in done_ids]
    print(f"[Етап 2] До завантаження: {len(todo):,} нових деталей")
    print(f"[Етап 2] Потоків паралельно: {N_WORKERS}")

    if not todo:
        return pd.DataFrame(rows)

    lock = Lock()
    counter = {"n": 0, "errors": 0}

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(fetch_one, mid): mid for mid in todo}

        with tqdm(total=len(todo), desc="Деталі (парал.)", unit=" mon") as pbar:
            for future in as_completed(futures):
                result = future.result()
                if result is not None:
                    rows.append(result)
                    counter["n"] += 1
                else:
                    counter["errors"] += 1

                pbar.update(1)
                pbar.set_postfix(saved=counter["n"], err=counter["errors"])

                # Періодичне збереження
                if counter["n"] > 0 and counter["n"] % SAVE_EVERY == 0:
                    with lock:
                        pd.DataFrame(rows).to_parquet(
                            OUTPUT_DETAILS, compression="snappy", engine="pyarrow"
                        )

    df = pd.DataFrame(rows)
    df.to_parquet(OUTPUT_DETAILS, compression="snappy", engine="pyarrow")
    print(f"\n[Етап 2] Збережено {len(df):,} моніторингів у {OUTPUT_DETAILS}")
    print(f"[Етап 2] Помилок: {counter['errors']:,}")
    return df


# ══════════════════════════════════════════════════════════════
# ПОБУДОВА МІТОК
# ══════════════════════════════════════════════════════════════

def build_tender_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Один тендер може мати кілька моніторингів — агрегуємо."""
    if df.empty:
        return pd.DataFrame()

    df_done = df[df["has_conclusion"] == True].copy()
    print(f"\n[Мітки] Моніторингів з висновком: {len(df_done):,}")

    def to_bool(x):
        if x is True or str(x).lower() == "true":
            return 1
        if x is False or str(x).lower() == "false":
            return 0
        return None

    df_done["v_occurred"] = df_done["violation_occurred"].apply(to_bool)

    agg = df_done.groupby("tender_id").agg(
        n_monitorings        =("monitoring_id",       "count"),
        n_with_violation     =("v_occurred",          "sum"),
        any_violation        =("v_occurred",          "max"),
        first_monitoring     =("date_created",        "min"),
        last_conclusion_date =("conclusion_date",     "max"),
        violation_types_all  =("violation_types",     lambda s: ";".join(
                                  v for v in s.dropna().unique() if v)),
        procuring_entity_id  =("procuring_entity_id", "first"),
        appeal_was_filed_any =("appeal_was_filed",    "max"),
        n_liabilities_sum    =("n_liabilities",       "sum"),
    ).reset_index()

    agg["dasu_label"] = agg["any_violation"].fillna(0).astype("int8")

    print(f"[Мітки] Унікальних тендерів з моніторингом: {len(agg):,}")
    print(f"        З порушеннями (label=1): "
          f"{agg['dasu_label'].sum():,} ({agg['dasu_label'].mean():.1%})")
    print(f"        Без порушень (label=0):  "
          f"{(agg['dasu_label']==0).sum():,}")

    return agg


# ══════════════════════════════════════════════════════════════
# СТАТИСТИКА
# ══════════════════════════════════════════════════════════════

def print_stats(df: pd.DataFrame):
    if df.empty:
        return
    print("\n" + "═" * 55)
    print("                  СТАТИСТИКА")
    print("═" * 55)
    print(f"Всього моніторингів:        {len(df):,}")
    print(f"\nЗа статусом:")
    print(df["status"].value_counts().to_string())
    print(f"\nЗ висновком:                {df['has_conclusion'].sum():,}")
    print(f"  з порушенням (True):      {(df['violation_occurred']==True).sum():,}")
    print(f"  без порушення (False):    {(df['violation_occurred']==False).sum():,}")

    if "date_created" in df.columns:
        dates = pd.to_datetime(df["date_created"], errors="coerce")
        if dates.notna().any():
            print(f"\nЧасовий діапазон:")
            print(f"  Перший:                   {dates.min()}")
            print(f"  Останній:                 {dates.max()}")

    print("═" * 55)


# ══════════════════════════════════════════════════════════════
# ГОЛОВНИЙ ПОТІК
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    t0 = time.time()

    print("═" * 55)
    print("  ЗАВАНТАЖЕННЯ МОНІТОРИНГІВ ДАСУ (ПАРАЛЕЛЬНО)")
    print("═" * 55)
    print(f"  API:        {API_BASE}")
    print(f"  Потоків:    {N_WORKERS}")
    print(f"  Сторінка:   {PAGE_LIMIT} моніторингів")
    print("═" * 55)

    # 1. Збираємо ID
    ids = collect_all_ids()

    # 2. Паралельно качаємо деталі
    df = fetch_details_parallel(ids)

    # 3. Статистика
    if not df.empty:
        print_stats(df)
        labels = build_tender_labels(df)
        if not labels.empty:
            labels.to_parquet(OUTPUT_LABELS, compression="snappy", engine="pyarrow")
            print(f"\n[Збережено] {OUTPUT_LABELS} — {len(labels):,} тендерів з мітками")

    elapsed = time.time() - t0
    print(f"\n✅ Готово за {elapsed/60:.1f} хвилин")
    print(f"   {OUTPUT_DETAILS}  — повні деталі моніторингів")
    print(f"   {OUTPUT_LABELS}   — мітки для JOIN з тендерами")
    print(f"   {IDS_CACHE}        — кеш ID моніторингів")

═══════════════════════════════════════════════════════
  ЗАВАНТАЖЕННЯ МОНІТОРИНГІВ ДАСУ (ПАРАЛЕЛЬНО)
═══════════════════════════════════════════════════════
  API:        https://audit-api.prozorro.gov.ua/api/2.5
  Потоків:    20
  Сторінка:   1000 моніторингів
═══════════════════════════════════════════════════════
[Етап 1] З кешу: 79,396 ID моніторингів
[Етап 2] З кешу: 79,396 моніторингів
[Етап 2] До завантаження: 0 нових деталей
[Етап 2] Потоків паралельно: 20

═══════════════════════════════════════════════════════
                  СТАТИСТИКА
═══════════════════════════════════════════════════════
Всього моніторингів:        79,396

За статусом:
status
addressed    52420
declined     15007
closed        6732
completed     4414
active         820
stopped          3

З висновком:                78,573
  з порушенням (True):      56,834
  без порушення (False):    21,739


ValueError: Mixed timezones detected. Pass utc=True in to_datetime or tz='UTC' in DatetimeIndex to convert to a common timezone.